# 02 LLM 候选重排序

本 notebook 只读取 `01_run_precompute.ipynb` 生成的 OAR Top-20 工作簿，并让 LLM 对全部 20 个候选执行完整排序。HPO 样本在 OAR Top-1 与 LCR 高排名候选之间存在局部父子或兄弟关系时，会携带代码生成的图谱上下文执行第二次完整排序。

它不会重新调用 embedding API。默认从 `results/precompute/` 读取预计算文件，并将结果保存到 `results/rerank/`。

## 1. 运行参数

只需要修改本单元。`EMBEDDING_MODEL` 用于定位对应的预计算文件。


In [ ]:
# 中文注释：选择数据集，可选 fgdd_phenotype、genereviews_10、bc8_t3、csc、gsc2017、gsc2024、id_68。
DATASET_KEY = "id_68"

# "FremyCompany/BioLORD-2023"  "dmis-lab/biobert-base-cased-v1.2"  "text-embedding-3-large"  "text-embedding-3-small"
# "FremyCompany/BioLORD-2023"
# "dmis-lab/biobert-base-cased-v1.2"
# "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"  # PubMedBERT
# "emilyalsentzer/Bio_ClinicalBERT"     # ClinicalBERT
# "text-embedding-3-large"
# "text-embedding-3-small"
EMBEDDING_MODEL = "text-embedding-3-large"

# 中文注释：与 01 notebook 一致，HPO 主流程默认读取带 _oar 标签的预计算结果。
USE_OAR_HPO_RETRIEVAL = True

# 中文注释：如需直接指定预计算工作簿可填写 Path 或字符串，否则按数据集和 embedding 模型自动定位。
PRECOMPUTE_PATH = None

# 中文注释：直接填写需要比较的具体 LLM 模型名称。
LLM_MODEL = "gpt-5.6-sol"

# 中文注释：None 表示读取 llm.max_concurrency，当前配置为 10。
LLM_MAX_CONCURRENCY = None

# 中文注释：所有并发线程共享该请求速率上限，给服务商的每分钟 100 次限制保留余量。
LLM_REQUESTS_PER_MINUTE = 200

# 中文注释：设置最终英文 prompt 使用的 LLM 采样温度。
LLM_TEMPERATURE = 0.0

# 中文注释：LLM 固定检查预计算前 20 个候选；方法结果只汇报 Accuracy@1。
LLM_CANDIDATE_COUNT = 20

# 中文注释：仅对 HPO 启用选择性图谱辅助，检查 LCR Top-3 与 OAR Top-1。
ENABLE_HPO_GRAPH_RERANK = True
HPO_GRAPH_PROBE_TOP_K = 3
HPO_MAX_ANCESTOR_DISTANCE = 2
HPO_GRAPH_TRIGGER_RULE = "dense_top1_disagrees_and_is_related_to_llm_top3"

# 中文注释：调试时可设为 10 或 100；全量运行使用 None。
MAX_SAMPLES = None

# 中文注释：失败请求最多重试三次；True 会复用与模型和候选严格匹配的检查点。
MAX_ATTEMPTS = 3
RESUME = True

# 中文注释：No Match 是纯文本返回值，不能启用只允许 JSON object 的响应格式。
USE_JSON_RESPONSE_FORMAT = False


## 2. 依赖、数据集和 ontology 配置


In [ ]:
# 中文注释：导入 LLM 调用、并发、校验、Excel 输出和 ontology 读取依赖。
import hashlib
import json
import re
import threading
import time
from collections import defaultdict, deque
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass
from itertools import combinations
from pathlib import Path
from typing import Any, Sequence

import numpy as np
import pandas as pd
from openai import OpenAI
from openpyxl import load_workbook
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.worksheet.table import Table, TableStyleInfo
from tqdm import tqdm

# 中文注释：从 notebook 所在目录分别定位预计算结果、重排序结果和检查点目录。
ROOT = Path.cwd().resolve()
CONFIG_PATH = ROOT / "LLM_config.json"
PRECOMPUTE_RESULTS_DIR = ROOT / "results" / "precompute"
RERANK_RESULTS_DIR = ROOT / "results" / "rerank"
CACHE_DIR = ROOT / ".cache" / "ontology_aligner"
PRECOMPUTE_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
RERANK_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# 中文注释：为当前八个测试集声明所属 ontology，其他列从预计算文件直接读取。
DATASETS = {
    "fgdd_phenotype": {"ontology": "hpo"},
    "fgdd_disease": {"ontology": "mondo"},
    "genereviews_10": {"ontology": "hpo"},
    "bc8_t3": {"ontology": "hpo"},
    "csc": {"ontology": "hpo"},
    "gsc2017": {"ontology": "hpo"},
    "gsc2024": {"ontology": "hpo"},
    "id_68": {"ontology": "hpo"},
}

# 中文注释：声明固定版本 HPO 和 MONDO JSONL 文件。
ONTOLOGIES = {
    "hpo": ROOT / "HPO/hp260623.jsonl",
    "mondo": ROOT / "MONDO/MONDO_terms_260706.jsonl",
}
HPO_GRAPH_PATH = ROOT / "HPO/hp260623.json"


## 3. LLM Prompt

只需在本单元修改最终英文 prompt；函数实现不包含 prompt 正文。


In [ ]:
# 中文注释：定义独立的系统提示词，用于约束医学 ontology 候选的完整重排序。
SYSTEM_PROMPT = """
You are an expert biomedical ontology curator performing phenotype concept normalization to Human Phenotype Ontology.

# Instructions
1. Rank every candidate concept from the best semantic match to the worst semantic match for the original biomedical text. 
2. Evaluate each candidate independently using its preferred name, synonyms, and definition. Prefer exact semantic equivalence and the most specific concept that fully captures the text over broader, narrower, or merely related concepts. 
3. Preserve distinctions such as anatomical site, clinical quality, severity, laterality, and disease versus phenotype.
4. Do not infer clinical details that are not stated in the original text, and do not treat the input candidate order as evidence.

# Output format
1. If at least one candidate is a valid semantic match, return exactly one JSON object in this form: {"ranking":["ID1","ID2"]}
2. The ranking array must contain every provided candidate ID exactly once, with no missing, duplicated, or additional IDs. 
3. If none of the candidates is a valid semantic match, return exactly the plain text No Match instead of JSON.
4. Return only the JSON object or No Match. Do not include explanations, scores, comments, or Markdown.
"""

# 中文注释：定义独立的用户提示词模板，运行时只替换 JSON 输入内容。
USER_PROMPT_TEMPLATE = """
The JSON below contains an original biomedical phrase and 20 candidate ontology concepts.

Rank all 20 candidates from best to worst according to how precisely and completely each concept represents the meaning of the original phrase.

{input_payload}

Return only the required JSON object with every candidate ID exactly once, or return exactly No Match if none is valid.
"""







# 中文注释：HGR 只处理已通过 LCR 匹配判定的样本，因此必须返回完整 20 项排序。
HPO_GRAPH_SYSTEM_PROMPT = """
You are an expert biomedical ontology curator refining phenotype concept normalization to Human Phenotype Ontology with hierarchy evidence.

# Instructions
1. Use the supplied HPO graph context together with preferred names, synonyms, and definitions to refine the ordering of every candidate concept. 
2. The graph context contains explicit is_a paths for highly similar HPO concepts. Do not infer clinical details that are not stated in the original text, and do not treat the input candidate order as evidence.

# Output format
1. Return exactly one JSON object in this form: {"ranking":["ID1","ID2"]}
2. The ranking array must contain every provided candidate ID exactly once, with no missing, duplicated, or additional IDs. 
3. Return JSON only. Do not include explanations, scores, comments, or Markdown.
"""

# 中文注释：定义 HPO 图谱辅助的用户提示词，运行时输入完整 20 个候选和代码生成的关系上下文。
HPO_GRAPH_RERANK_PROMPT = """
The JSON below contains an original biomedical phrase, 20 candidate HPO concepts, and additional HPO graph context for several highly similar concepts.

Rank all 20 candidates from best to worst according to how precisely and completely each concept represents the meaning of the original phrase.

Some candidates are highly similar. Pay particular attention to the HPO graph relationships and definitions provided for these concepts, and use this information to distinguish them carefully.

{input_payload}

Return only the required JSON object. The ranking must include every candidate ID exactly once, ordered from the best match to the worst match.
"""

## 4. 基础工具与 ontology 读取


In [ ]:
# 中文注释：统一保存 HPO 或 MONDO 候选的名称、同义词和定义。
@dataclass(frozen=True)
class OntologyTerm:
    ontology_id: str
    name: str
    synonyms: tuple[str, ...]
    definition: str


# 中文注释：保存固定 HPO 版本的父节点索引以及图中节点的名称和定义。
@dataclass(frozen=True)
class HPOGraphIndex:
    parents: dict[str, frozenset[str]]
    names: dict[str, str]
    definitions: dict[str, str]


# 中文注释：读取本地 API 配置但不显示密钥。
def load_config(path: Path) -> dict[str, Any]:
    with path.open("r", encoding="utf-8-sig") as handle:
        return json.load(handle)


# 中文注释：计算预计算工作簿哈希，用于严格隔离 LLM 检查点。
def file_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


# 中文注释：把模型名转换成稳定文件标签。
def safe_tag(value: str, limit: int = 48) -> str:
    tag = re.sub(r"[^A-Za-z0-9._-]+", "_", value).strip("._-").lower()
    return (tag or "model")[:limit]


# 中文注释：解析 MONDO 同义词字符串开头的引号内容。
def parse_mondo_synonym(value: str) -> str:
    match = re.match(r'^"((?:\\.|[^"\\])*)"', str(value or ""))
    if not match:
        return str(value or "").strip()
    return json.loads(f'"{match.group(1)}"')


# 中文注释：读取固定版本 HPO 或 MONDO 候选详情。
def load_ontology(ontology_key: str) -> dict[str, OntologyTerm]:
    terms: dict[str, OntologyTerm] = {}
    with ONTOLOGIES[ontology_key].open("r", encoding="utf-8-sig") as handle:
        for line in handle:
            if not line.strip():
                continue
            item = json.loads(line)
            if ontology_key == "hpo":
                if item.get("is_obsolete"):
                    continue
                ontology_id = str(item["HPOid"])
                synonyms = tuple(str(entry["text"]) for entry in item.get("synonyms", []))
                definition = str(item.get("definition") or "")
            else:
                ontology_id = str(item["MONDO_id"])
                synonyms = tuple(parse_mondo_synonym(entry) for entry in item.get("synonym", []))
                definition = str(item.get("description") or "").strip('"')
            terms[ontology_id] = OntologyTerm(
                ontology_id=ontology_id,
                name=str(item["name"]),
                synonyms=tuple(dict.fromkeys(text for text in synonyms if text)),
                definition=definition,
            )
    return terms


# 中文注释：把 HPO 图 JSON 中的 URI 转换为标准 HP:nnnnnnn ID。
def normalize_hpo_uri(value: str) -> str:
    local_id = str(value).rsplit("/", 1)[-1]
    return local_id.replace("HP_", "HP:", 1)


# 中文注释：直接读取固定 HPO JSON，并在内存中建立父节点、名称和定义索引。
def load_hpo_graph(path: Path) -> HPOGraphIndex:
    with path.open("r", encoding="utf-8-sig") as handle:
        graph = json.load(handle)["graphs"][0]
    parents: dict[str, set[str]] = defaultdict(set)
    names: dict[str, str] = {}
    definitions: dict[str, str] = {}
    for node in graph.get("nodes", []):
        ontology_id = normalize_hpo_uri(node["id"])
        meta = node.get("meta") or {}
        definition_meta = meta.get("definition") or {}
        names[ontology_id] = str(node.get("lbl") or ontology_id)
        definitions[ontology_id] = str(definition_meta.get("val") or "")
    for edge in graph.get("edges", []):
        if edge.get("pred") != "is_a":
            continue
        child_id = normalize_hpo_uri(edge["sub"])
        parent_id = normalize_hpo_uri(edge["obj"])
        parents[child_id].add(parent_id)
    return HPOGraphIndex(
        parents={ontology_id: frozenset(values) for ontology_id, values in parents.items()},
        names=names,
        definitions=definitions,
    )


# 中文注释：沿 is_a 父节点方向查找限定层数内的最短祖先路径。
def find_is_a_path(
    child_id: str,
    ancestor_id: str,
    graph: HPOGraphIndex,
    max_edges: int,
) -> list[str]:
    queue = deque([(child_id, [child_id])])
    visited = {child_id}
    while queue:
        current_id, path = queue.popleft()
        if len(path) - 1 >= max_edges:
            continue
        for parent_id in sorted(graph.parents.get(current_id, ())):
            next_path = [*path, parent_id]
            if parent_id == ancestor_id:
                return next_path
            if parent_id not in visited:
                visited.add(parent_id)
                queue.append((parent_id, next_path))
    return []


# 中文注释：从固定 ontology 信息生成图谱上下文中的名称和定义。
def graph_concept_payload(
    ontology_id: str,
    terms: dict[str, OntologyTerm],
    graph: HPOGraphIndex,
) -> dict[str, str]:
    term = terms.get(ontology_id)
    name = term.name if term else graph.names.get(ontology_id, ontology_id)
    definition = term.definition if term else graph.definitions.get(ontology_id, "")
    return {
        "id": ontology_id,
        "preferred_name": name,
        "definition": definition or "No definition provided in this HPO release.",
    }


# 中文注释：把一条 is_a 边展开为包含两端名称的结构化关系。
def is_a_relation_payload(
    child_id: str,
    parent_id: str,
    terms: dict[str, OntologyTerm],
    graph: HPOGraphIndex,
) -> dict[str, str]:
    child = graph_concept_payload(child_id, terms, graph)
    parent = graph_concept_payload(parent_id, terms, graph)
    return {
        "subject_id": child_id,
        "subject_name": child["preferred_name"],
        "predicate": "is_a",
        "object_id": parent_id,
        "object_name": parent["preferred_name"],
    }


# 中文注释：仅在 LCR 与 OAR Top-1 不一致且存在局部关系时，生成父子或直接兄弟图谱上下文。
def build_hpo_graph_context(
    llm_ranking: Sequence[str],
    dense_top1_id: str,
    terms: dict[str, OntologyTerm],
    graph: HPOGraphIndex,
) -> list[dict[str, Any]]:
    if not llm_ranking or llm_ranking[0] == dense_top1_id:
        return []
    probe_ids = list(dict.fromkeys([*llm_ranking[:HPO_GRAPH_PROBE_TOP_K], dense_top1_id]))
    contexts: list[dict[str, Any]] = []
    for left_id, right_id in combinations(probe_ids, 2):
        if dense_top1_id not in {left_id, right_id}:
            continue
        path = find_is_a_path(left_id, right_id, graph, HPO_MAX_ANCESTOR_DISTANCE)
        if not path:
            reverse_path = find_is_a_path(right_id, left_id, graph, HPO_MAX_ANCESTOR_DISTANCE)
            path = reverse_path
        if path:
            descendant = graph_concept_payload(path[0], terms, graph)
            ancestor = graph_concept_payload(path[-1], terms, graph)
            contexts.append({
                "related_concepts": [descendant, ancestor],
                "relationship_explanation": (
                    f"{descendant['preferred_name']} ({descendant['id']}) is a more specific "
                    f"descendant of {ancestor['preferred_name']} ({ancestor['id']}) in HPO. "
                    "The relationship is represented by the following is_a path."
                ),
                "is_a_relations": [
                    is_a_relation_payload(child_id, parent_id, terms, graph)
                    for child_id, parent_id in zip(path, path[1:])
                ],
                "concept_definitions": [
                    graph_concept_payload(ontology_id, terms, graph) for ontology_id in path
                ],
            })
            continue
        shared_parent_ids = sorted(
            graph.parents.get(left_id, frozenset()) & graph.parents.get(right_id, frozenset())
        )
        if not shared_parent_ids:
            continue
        left = graph_concept_payload(left_id, terms, graph)
        right = graph_concept_payload(right_id, terms, graph)
        shared_names = [graph_concept_payload(parent_id, terms, graph)["preferred_name"] for parent_id in shared_parent_ids]
        relation_nodes = [left_id, right_id, *shared_parent_ids]
        contexts.append({
            "related_concepts": [left, right],
            "relationship_explanation": (
                f"{left['preferred_name']} ({left_id}) and {right['preferred_name']} ({right_id}) "
                f"are sibling HPO concepts because they share the following direct parent(s): "
                f"{', '.join(shared_names)}."
            ),
            "is_a_relations": [
                *[is_a_relation_payload(left_id, parent_id, terms, graph) for parent_id in shared_parent_ids],
                *[is_a_relation_payload(right_id, parent_id, terms, graph) for parent_id in shared_parent_ids],
            ],
            "concept_definitions": [
                graph_concept_payload(ontology_id, terms, graph) for ontology_id in relation_nodes
            ],
        })
    return contexts


# 中文注释：读取预计算 Samples 和 Run_Config，并校验 Top-20、数据集和 ontology。
def load_precompute(path: Path, dataset_key: str) -> tuple[pd.DataFrame, dict[str, Any]]:
    samples = pd.read_excel(path, sheet_name="Samples", dtype=str).fillna("")
    config_frame = pd.read_excel(path, sheet_name="Run_Config", dtype=str).fillna("")
    run_config = dict(zip(config_frame["parameter"], config_frame["value"]))
    required = {"sample_id", "source_excel_row", "query", "preferred_name", "gold_id", "accepted_gold_ids_json", "candidate_ids_json"}
    missing = required - set(samples.columns)
    if missing:
        raise KeyError(f"预计算文件缺少列: {sorted(missing)}")
    if run_config.get("dataset_key") != dataset_key:
        raise ValueError(f"预计算数据集不匹配: {run_config.get('dataset_key')} != {dataset_key}")
    if int(float(run_config.get("retrieval_top_k", 0))) != LLM_CANDIDATE_COUNT:
        raise ValueError("LLM 重排序要求 01 notebook 的 Top-20 预计算文件")
    if MAX_SAMPLES is not None:
        samples = samples.head(int(MAX_SAMPLES)).copy()
    return samples, run_config


## 5. LLM 请求、解析与检查点


In [ ]:
# 中文注释：在全部并发线程之间平滑分配 API 请求许可，并在限流时统一暂停。
class GlobalRequestRateLimiter:
    # 中文注释：把每分钟请求上限换算成相邻请求的最小启动间隔。
    def __init__(self, requests_per_minute: int):
        if int(requests_per_minute) <= 0:
            raise ValueError("LLM_REQUESTS_PER_MINUTE 必须大于 0")
        self.requests_per_minute = int(requests_per_minute)
        self.interval_seconds = 60.0 / self.requests_per_minute
        self._condition = threading.Condition()
        self._next_request_at = 0.0
        self._blocked_until = 0.0

    # 中文注释：线程只有取得全局时间槽后才能真正调用 API。
    def acquire(self) -> None:
        with self._condition:
            while True:
                now = time.monotonic()
                ready_at = max(self._next_request_at, self._blocked_until)
                wait_seconds = ready_at - now
                if wait_seconds <= 0:
                    self._next_request_at = now + self.interval_seconds
                    return
                self._condition.wait(timeout=wait_seconds)

    # 中文注释：任一线程收到 429 后延长全局冷却时间，阻止其他线程继续冲击额度。
    def pause(self, seconds: float) -> None:
        with self._condition:
            self._blocked_until = max(self._blocked_until, time.monotonic() + float(seconds))
            self._next_request_at = max(self._next_request_at, self._blocked_until)
            self._condition.notify_all()


# 中文注释：为每个工作线程创建独立 OpenAI client，并共享同一个请求限速器。
class LLMReranker:
    # 中文注释：初始化模型、API 地址、重试参数和线程本地状态。
    def __init__(
        self,
        config: dict[str, Any],
        model: str,
        max_attempts: int,
        requests_per_minute: int,
    ):
        self.api_key = config["api_key"]
        self.base_url = config.get("base_url") or None
        self.model = model
        self.max_attempts = int(max_attempts)
        self.rate_limiter = GlobalRequestRateLimiter(requests_per_minute)
        self._thread_local = threading.local()

    # 中文注释：按线程创建 client，避免并发请求共享可变状态。
    def _client(self) -> OpenAI:
        if not hasattr(self._thread_local, "client"):
            kwargs = {"api_key": self.api_key, "max_retries": 0}
            if self.base_url:
                kwargs["base_url"] = self.base_url
            self._thread_local.client = OpenAI(**kwargs)
        return self._thread_local.client

    # 中文注释：解析完整候选排列或严格的 No Match 纯文本结果。
    @staticmethod
    def _parse_ranking(content: str, allowed_ids: Sequence[str]) -> tuple[list[str], bool]:
        cleaned = re.sub(r"^```(?:json)?\s*|\s*```$", "", str(content or "").strip(), flags=re.IGNORECASE)
        if cleaned == "No Match" or cleaned == '"No Match"':
            return [], True
        payload = json.loads(cleaned)
        ranking = payload.get("ranking") if isinstance(payload, dict) else None
        if not isinstance(ranking, list):
            raise ValueError("LLM 返回值缺少 ranking 数组")
        ranking = [str(value) for value in ranking]
        if len(ranking) != len(allowed_ids) or set(ranking) != set(allowed_ids):
            raise ValueError("ranking 不是候选 ID 的完整排列")
        return ranking, False

    # 中文注释：把完整候选列表转换为两次 LLM 调用共用的结构化输入。
    @staticmethod
    def _candidate_payloads(candidates: Sequence[OntologyTerm]) -> list[dict[str, Any]]:
        return [
            {
                "id": term.ontology_id,
                "preferred_name": term.name,
                "synonyms": list(term.synonyms[:20]),
                "definition": term.definition[:2000],
            }
            for term in candidates
        ]

    # 中文注释：执行一次受约束的 LLM 排序请求，并统一记录重试、token 和耗时。
    def _request_ranking(
        self,
        system_prompt: str,
        user_prompt: str,
        allowed_ids: Sequence[str],
        allow_no_match: bool,
    ) -> dict[str, Any]:
        last_error = ""
        for attempt in range(1, self.max_attempts + 1):
            try:
                kwargs: dict[str, Any] = {
                    "model": self.model,
                    "temperature": LLM_TEMPERATURE,
                    "messages": [
                        {"role": "system", "content": system_prompt},
                        {"role": "user", "content": user_prompt},
                    ],
                }
                if USE_JSON_RESPONSE_FORMAT:
                    kwargs["response_format"] = {"type": "json_object"}
                self.rate_limiter.acquire()
                started = time.perf_counter()
                response = self._client().chat.completions.create(**kwargs)
                content = response.choices[0].message.content or ""
                ranking, no_match = self._parse_ranking(content, allowed_ids)
                if no_match and not allow_no_match:
                    raise ValueError("当前任务配置未启用 No Match")
                usage = response.usage
                return {
                    "ranking": ranking,
                    "no_match": no_match,
                    "attempts": attempt,
                    "api_seconds": round(time.perf_counter() - started, 6),
                    "prompt_tokens": int(getattr(usage, "prompt_tokens", 0) or 0),
                    "completion_tokens": int(getattr(usage, "completion_tokens", 0) or 0),
                    "total_tokens": int(getattr(usage, "total_tokens", 0) or 0),
                    "raw_response": content,
                    "error": "",
                }
            except Exception as error:
                last_error = f"{type(error).__name__}: {error}"
                if getattr(error, "status_code", None) == 429:
                    self.rate_limiter.pause(60.0)
                if attempt < self.max_attempts:
                    time.sleep(2 ** attempt)
        return {
            "ranking": [],
            "no_match": False,
            "attempts": self.max_attempts,
            "api_seconds": 0.0,
            "prompt_tokens": 0,
            "completion_tokens": 0,
            "total_tokens": 0,
            "raw_response": "",
            "error": last_error,
        }

    # 中文注释：第一次调用使用普通 prompt 对完整 20 个候选排序，并允许返回 No Match。
    def rank(self, query: str, candidates: Sequence[OntologyTerm]) -> dict[str, Any]:
        allowed_ids = [term.ontology_id for term in candidates]
        input_payload = json.dumps(
            {"original_text": query, "candidates": self._candidate_payloads(candidates)},
            ensure_ascii=False,
            indent=2,
        )
        user_prompt = USER_PROMPT_TEMPLATE.format(input_payload=input_payload)
        return self._request_ranking(SYSTEM_PROMPT, user_prompt, allowed_ids, allow_no_match=True)

    # 中文注释：第二次 HGR 调用只重排 LCR 已确认可匹配的样本，不允许返回 No Match。
    def rank_with_hpo_graph(
        self,
        query: str,
        candidates: Sequence[OntologyTerm],
        graph_context: Sequence[dict[str, Any]],
    ) -> dict[str, Any]:
        allowed_ids = [term.ontology_id for term in candidates]
        input_payload = json.dumps(
            {
                "original_text": query,
                "candidates": self._candidate_payloads(candidates),
                "hpo_graph_context": list(graph_context),
            },
            ensure_ascii=False,
            indent=2,
        )
        user_prompt = HPO_GRAPH_RERANK_PROMPT.format(input_payload=input_payload)
        return self._request_ranking(
            HPO_GRAPH_SYSTEM_PROMPT, user_prompt, allowed_ids, allow_no_match=False
        )


# 中文注释：生成候选指纹，确保检查点不能跨预计算结果错误复用。
def candidate_fingerprint(candidate_ids: Sequence[str]) -> str:
    return hashlib.sha256("|".join(candidate_ids).encode("utf-8")).hexdigest()


# 中文注释：读取与模型、查询和候选指纹完全一致的 LLM 检查点。
def load_checkpoint(
    path: Path,
    samples_by_id: dict[int, dict[str, Any]],
    model: str,
    rerank_config_hash: str,
) -> dict[int, dict[str, Any]]:
    completed: dict[int, dict[str, Any]] = {}
    if not RESUME or not path.exists():
        return completed
    with path.open("r", encoding="utf-8-sig") as handle:
        for line in handle:
            if not line.strip():
                continue
            try:
                item = json.loads(line)
            except json.JSONDecodeError:
                continue
            sample = samples_by_id.get(int(item.get("sample_id", -1)))
            if sample is None or item.get("model") != model:
                continue
            if item.get("rerank_config_hash") != rerank_config_hash:
                continue
            candidate_ids = json.loads(sample["candidate_ids_json"])[:LLM_CANDIDATE_COUNT]
            if item.get("query") != sample["query"]:
                continue
            if item.get("accepted_gold_ids") != json.loads(sample["accepted_gold_ids_json"]):
                continue
            if item.get("candidate_fingerprint") != candidate_fingerprint(candidate_ids):
                continue
            checkpoint_ranking = item.get("ranking", [])
            ranking_is_complete = (
                isinstance(checkpoint_ranking, list)
                and len(checkpoint_ranking) == len(candidate_ids)
                and set(checkpoint_ranking) == set(candidate_ids)
            )
            result_is_no_match = item.get("no_match") is True and item.get("ranking", []) == []
            if not item.get("error") and (ranking_is_complete or result_is_no_match):
                completed[int(item["sample_id"])] = item
    return completed


# 中文注释：主线程追加单条 LLM 检查点，避免并发写入损坏 JSONL。
def append_checkpoint(path: Path, record: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(record, ensure_ascii=False, separators=(",", ":")) + "\n")


# 中文注释：为一个样本执行普通排序，并在检测到 HPO 近邻关系时执行第二次完整排序。
def rerank_one(
    sample: dict[str, Any],
    terms: dict[str, OntologyTerm],
    hpo_graph: HPOGraphIndex | None,
    reranker: LLMReranker,
    model: str,
    rerank_config_hash: str,
) -> dict[str, Any]:
    candidate_ids = json.loads(sample["candidate_ids_json"])[:LLM_CANDIDATE_COUNT]
    if len(candidate_ids) != LLM_CANDIDATE_COUNT or len(set(candidate_ids)) != LLM_CANDIDATE_COUNT:
        raise ValueError(f"sample_id={sample['sample_id']} 没有 20 个唯一候选")
    missing = [ontology_id for ontology_id in candidate_ids if ontology_id not in terms]
    if missing:
        raise KeyError(f"sample_id={sample['sample_id']} 的候选不在固定 ontology 中: {missing[:3]}")
    candidate_terms = [terms[ontology_id] for ontology_id in candidate_ids]
    initial_result = reranker.rank(sample["query"], candidate_terms)
    graph_context: list[dict[str, Any]] = []
    graph_result: dict[str, Any] | None = None
    if hpo_graph is not None and not initial_result["error"] and not initial_result["no_match"]:
        dense_top1_id = candidate_ids[0]
        graph_context = build_hpo_graph_context(
            initial_result["ranking"], dense_top1_id, terms, hpo_graph
        )
        if graph_context:
            graph_candidate_terms = [terms[ontology_id] for ontology_id in initial_result["ranking"]]
            graph_result = reranker.rank_with_hpo_graph(
                sample["query"], graph_candidate_terms, graph_context
            )
    final_result = graph_result if graph_result is not None else initial_result
    graph_attempts = int(graph_result.get("attempts", 0)) if graph_result else 0
    graph_api_seconds = float(graph_result.get("api_seconds", 0.0)) if graph_result else 0.0
    graph_prompt_tokens = int(graph_result.get("prompt_tokens", 0)) if graph_result else 0
    graph_completion_tokens = int(graph_result.get("completion_tokens", 0)) if graph_result else 0
    graph_total_tokens = int(graph_result.get("total_tokens", 0)) if graph_result else 0
    return {
        "sample_id": int(sample["sample_id"]),
        "source_excel_row": int(sample["source_excel_row"]),
        "query": sample["query"],
        "preferred_name": sample["preferred_name"],
        "gold_id": sample["gold_id"],
        "accepted_gold_ids": json.loads(sample["accepted_gold_ids_json"]),
        "candidate_fingerprint": candidate_fingerprint(candidate_ids),
        "model": model,
        "rerank_config_hash": rerank_config_hash,
        "ranking": final_result["ranking"],
        "no_match": final_result["no_match"],
        "graph_rerank_applied": graph_result is not None,
        "graph_context_count": len(graph_context),
        "hpo_graph_context": graph_context,
        "initial_ranking": initial_result["ranking"],
        "initial_no_match": initial_result["no_match"],
        "initial_attempts": initial_result["attempts"],
        "graph_attempts": graph_attempts,
        "attempts": int(initial_result["attempts"]) + graph_attempts,
        "initial_api_seconds": initial_result["api_seconds"],
        "graph_api_seconds": graph_api_seconds,
        "api_seconds": round(float(initial_result["api_seconds"]) + graph_api_seconds, 6),
        "initial_prompt_tokens": initial_result["prompt_tokens"],
        "graph_prompt_tokens": graph_prompt_tokens,
        "prompt_tokens": int(initial_result["prompt_tokens"]) + graph_prompt_tokens,
        "initial_completion_tokens": initial_result["completion_tokens"],
        "graph_completion_tokens": graph_completion_tokens,
        "completion_tokens": int(initial_result["completion_tokens"]) + graph_completion_tokens,
        "initial_total_tokens": initial_result["total_tokens"],
        "graph_total_tokens": graph_total_tokens,
        "total_tokens": int(initial_result["total_tokens"]) + graph_total_tokens,
        "initial_raw_response": initial_result["raw_response"],
        "graph_raw_response": graph_result.get("raw_response", "") if graph_result else "",
        "raw_response": final_result["raw_response"],
        "error": final_result["error"],
    }


## 6. 结果整理与 Excel 输出


In [ ]:
# 中文注释：把 LLM 全排列或 No Match 状态展开为单样本一行，并只计算 Accuracy@1。
def build_output_frame(records: Sequence[dict[str, Any]], terms: dict[str, OntologyTerm]) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    for record in sorted(records, key=lambda item: int(item["sample_id"])):
        ranking = record.get("ranking", [])
        initial_ranking = record.get("initial_ranking", ranking)
        graph_context = record.get("hpo_graph_context", [])
        no_match = record.get("no_match") is True
        gold_id = record["gold_id"]
        accepted_ids = tuple(record.get("accepted_gold_ids", [gold_id]))
        if gold_id not in accepted_ids:
            raise ValueError(f"当前 Gold ID 不在 accepted_gold_ids 中: {gold_id}")
        accepted_set = set(accepted_ids)
        gold_rank = next((rank for rank, ontology_id in enumerate(ranking, 1) if ontology_id in accepted_set), None)
        row: dict[str, Any] = {
            "sample_id": record["sample_id"],
            "source_excel_row": record["source_excel_row"],
            "query": record["query"],
            "preferred_name": record["preferred_name"],
            "gold_id": gold_id,
            "accepted_gold_ids_json": json.dumps(accepted_ids, ensure_ascii=False, separators=(",", ":")),
            "prediction_status": "no_match" if no_match else "ranked",
            "graph_rerank_applied": bool(record.get("graph_rerank_applied", False)),
            "graph_context_count": int(record.get("graph_context_count", 0)),
            "initial_top1_id": initial_ranking[0] if initial_ranking else "No Match",
            "initial_top1_name": terms[initial_ranking[0]].name if initial_ranking else "",
            "predicted_top1_id": ranking[0] if ranking else "No Match",
            "predicted_top1_name": terms[ranking[0]].name if ranking else "",
            "gold_llm_rank": gold_rank,
            "hit_at_1": bool(gold_rank and gold_rank <= 1),
            "ranked_ids_json": json.dumps(ranking, ensure_ascii=False, separators=(",", ":")),
            "initial_ranked_ids_json": json.dumps(initial_ranking, ensure_ascii=False, separators=(",", ":")),
            "hpo_graph_context_json": json.dumps(graph_context, ensure_ascii=False, separators=(",", ":")),
            "initial_attempts": record.get("initial_attempts", record.get("attempts", 0)),
            "graph_attempts": record.get("graph_attempts", 0),
            "attempts": record.get("attempts", 0),
            "initial_api_seconds": record.get("initial_api_seconds", record.get("api_seconds", 0.0)),
            "graph_api_seconds": record.get("graph_api_seconds", 0.0),
            "api_seconds": record.get("api_seconds", 0.0),
            "initial_prompt_tokens": record.get("initial_prompt_tokens", record.get("prompt_tokens", 0)),
            "graph_prompt_tokens": record.get("graph_prompt_tokens", 0),
            "prompt_tokens": record.get("prompt_tokens", 0),
            "initial_completion_tokens": record.get("initial_completion_tokens", record.get("completion_tokens", 0)),
            "graph_completion_tokens": record.get("graph_completion_tokens", 0),
            "completion_tokens": record.get("completion_tokens", 0),
            "initial_total_tokens": record.get("initial_total_tokens", record.get("total_tokens", 0)),
            "graph_total_tokens": record.get("graph_total_tokens", 0),
            "total_tokens": record.get("total_tokens", 0),
            "error": record.get("error", ""),
        }
        for rank, ontology_id in enumerate(ranking, 1):
            row[f"rank_{rank:02d}_id"] = ontology_id
            row[f"rank_{rank:02d}_name"] = terms[ontology_id].name
        rows.append(row)
    return pd.DataFrame(rows)


# 中文注释：为结果表添加筛选、冻结窗格、列宽和条纹样式。
def style_sheet(workbook_path: Path, sheet_name: str, widths: dict[str, float]) -> None:
    workbook = load_workbook(workbook_path)
    sheet = workbook[sheet_name]
    sheet.freeze_panes = "A2"
    sheet.sheet_view.showGridLines = False
    for column, width in widths.items():
        sheet.column_dimensions[column].width = width
    for cell in sheet[1]:
        cell.fill = PatternFill("solid", fgColor="176567")
        cell.font = Font(color="FFFFFF", bold=True)
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    if sheet.max_row >= 2 and sheet.max_column >= 1:
        table = Table(displayName=f"{sheet_name.replace('_', '')}Table", ref=sheet.dimensions)
        table.tableStyleInfo = TableStyleInfo(
            name="TableStyleMedium2",
            showFirstColumn=False,
            showLastColumn=False,
            showRowStripes=True,
            showColumnStripes=False,
        )
        sheet.add_table(table)
    workbook.save(workbook_path)


# 中文注释：写出 Rankings 和 Run_Config，不创建 Summary sheet。
def export_reranking(output_path: Path, output_frame: pd.DataFrame, run_config: dict[str, Any]) -> None:
    config_frame = pd.DataFrame([{"parameter": key, "value": value} for key, value in run_config.items()])
    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        output_frame.to_excel(writer, sheet_name="Rankings", index=False)
        config_frame.to_excel(writer, sheet_name="Run_Config", index=False)
    style_sheet(
        output_path,
        "Rankings",
        {"A": 12, "B": 16, "C": 42, "D": 30, "E": 18, "F": 60, "G": 18, "H": 18, "I": 18, "J": 18, "K": 30, "L": 18, "M": 30},
    )
    style_sheet(output_path, "Run_Config", {"A": 36, "B": 100})


## 7. 执行 LLM 全量排序与选择性 HPO 图谱辅助

运行本单元会调用 LLM API；检查点允许中断后继续。


In [ ]:
# 中文注释：解析数据集、embedding 预计算文件和 LLM 参数。
if DATASET_KEY not in DATASETS:
    raise KeyError(f"未知 DATASET_KEY: {DATASET_KEY}")
if LLM_CANDIDATE_COUNT != 20:
    raise ValueError("本项目 LLM 排序固定使用前 20 个候选")
if not 1 <= HPO_GRAPH_PROBE_TOP_K <= LLM_CANDIDATE_COUNT:
    raise ValueError("HPO_GRAPH_PROBE_TOP_K 必须在 1 到 20 之间")
if HPO_MAX_ANCESTOR_DISTANCE not in {1, 2}:
    raise ValueError("HPO_MAX_ANCESTOR_DISTANCE 只支持 1 或 2")
if USE_JSON_RESPONSE_FORMAT:
    raise ValueError("No Match 是纯文本结果，USE_JSON_RESPONSE_FORMAT 必须为 False")
config = load_config(CONFIG_PATH)
llm_config = config["llm"]
resolved_embedding_model = str(EMBEDDING_MODEL)
resolved_llm_model = str(LLM_MODEL)
resolved_concurrency = int(LLM_MAX_CONCURRENCY or llm_config.get("max_concurrency", 1))
use_oar = bool(USE_OAR_HPO_RETRIEVAL and DATASETS[DATASET_KEY]["ontology"] == "hpo" and resolved_embedding_model == "text-embedding-3-large")
embedding_tag = safe_tag(f"{resolved_embedding_model}_oar") if use_oar else safe_tag(resolved_embedding_model)
expected_retrieval_method = "oar_dense_cosine" if use_oar else "dense_cosine"
llm_tag = safe_tag(resolved_llm_model)
precompute_path = Path(PRECOMPUTE_PATH) if PRECOMPUTE_PATH else PRECOMPUTE_RESULTS_DIR / f"{DATASET_KEY}_retrieval_top20_{embedding_tag}_dense-only.xlsx"
if not precompute_path.exists():
    raise FileNotFoundError(f"找不到预计算文件，请先运行 01 notebook: {precompute_path}")

# 中文注释：读取 Top-20 文件和固定 ontology，LLM 使用每条记录的全部 20 个候选。
samples_frame, precompute_config = load_precompute(precompute_path, DATASET_KEY)
if precompute_config.get("retrieval_method") != expected_retrieval_method:
    raise ValueError(
        f"预计算检索方法不匹配: {precompute_config.get('retrieval_method')} "
        f"!= {expected_retrieval_method}; 必须先重新运行 01 notebook"
    )
ontology_key = DATASETS[DATASET_KEY]["ontology"]
terms = load_ontology(ontology_key)
hpo_graph_enabled = bool(ENABLE_HPO_GRAPH_RERANK and ontology_key == "hpo")
if hpo_graph_enabled and not HPO_GRAPH_PATH.exists():
    raise FileNotFoundError(f"找不到固定 HPO 图文件: {HPO_GRAPH_PATH}")
hpo_graph = load_hpo_graph(HPO_GRAPH_PATH) if hpo_graph_enabled else None
hpo_graph_sha256 = file_sha256(HPO_GRAPH_PATH) if hpo_graph_enabled else ""
samples = samples_frame.to_dict(orient="records")
samples_by_id = {int(sample["sample_id"]): sample for sample in samples}
precompute_hash = file_sha256(precompute_path)

# 中文注释：把温度、候选数和 prompt 纳入哈希，防止配置变化后错误复用旧检查点。
rerank_config_identity = {
    "llm_model": resolved_llm_model,
    "llm_temperature": LLM_TEMPERATURE,
    "llm_candidate_count": LLM_CANDIDATE_COUNT,
    "use_json_response_format": USE_JSON_RESPONSE_FORMAT,
    "system_prompt_sha256": hashlib.sha256(SYSTEM_PROMPT.encode("utf-8")).hexdigest(),
    "user_prompt_template_sha256": hashlib.sha256(USER_PROMPT_TEMPLATE.encode("utf-8")).hexdigest(),
    "hpo_graph_rerank_enabled": hpo_graph_enabled,
    "hpo_graph_allow_no_match": False,
    "hpo_graph_probe_top_k": HPO_GRAPH_PROBE_TOP_K,
    "hpo_max_ancestor_distance": HPO_MAX_ANCESTOR_DISTANCE,
    "hpo_graph_trigger_rule": HPO_GRAPH_TRIGGER_RULE,
    "hpo_graph_sha256": hpo_graph_sha256,
    "hpo_graph_system_prompt_sha256": hashlib.sha256(HPO_GRAPH_SYSTEM_PROMPT.encode("utf-8")).hexdigest(),
    "hpo_graph_user_prompt_sha256": hashlib.sha256(HPO_GRAPH_RERANK_PROMPT.encode("utf-8")).hexdigest(),
}
rerank_config_payload = json.dumps(rerank_config_identity, ensure_ascii=True, sort_keys=True, separators=(",", ":"))
rerank_config_hash = hashlib.sha256(rerank_config_payload.encode("utf-8")).hexdigest()[:12]
checkpoint_path = (
    CACHE_DIR / "llm_checkpoints"
    / f"{DATASET_KEY}_{embedding_tag}_{llm_tag}_{precompute_hash[:12]}_{rerank_config_hash}.jsonl"
)
completed = load_checkpoint(checkpoint_path, samples_by_id, resolved_llm_model, rerank_config_hash)
pending = [sample for sample in samples if int(sample["sample_id"]) not in completed]
reranker = LLMReranker(
    llm_config, resolved_llm_model, MAX_ATTEMPTS, LLM_REQUESTS_PER_MINUTE
)
print(
    f"samples={len(samples)}; completed={len(completed)}; pending={len(pending)}; "
    f"llm={resolved_llm_model}; concurrency={resolved_concurrency}; "
    f"rpm_limit={LLM_REQUESTS_PER_MINUTE}; hpo_graph={hpo_graph_enabled}"
)

# 中文注释：以配置并发执行 LLM 排序，逐条落盘后用 tqdm 更新成功和失败进度。
started_at = time.perf_counter()
failed_this_run = 0
graph_reranks_this_run = 0
with tqdm(
    total=len(samples),
    initial=len(completed),
    desc="LLM rerank",
    unit="sample",
    dynamic_ncols=True,
    mininterval=0.5,
) as progress, ThreadPoolExecutor(max_workers=resolved_concurrency) as executor:
    futures = {
        executor.submit(
            rerank_one, sample, terms, hpo_graph, reranker, resolved_llm_model, rerank_config_hash
        ): int(sample["sample_id"])
        for sample in pending
    }
    for future in as_completed(futures):
        record = future.result()
        append_checkpoint(checkpoint_path, record)
        if not record["error"]:
            completed[int(record["sample_id"])] = record
        else:
            failed_this_run += 1
        if record.get("graph_rerank_applied"):
            graph_reranks_this_run += 1
        progress.update(1)
        progress.set_postfix(
            completed=len(completed), failed=failed_this_run, graph=graph_reranks_this_run, refresh=False
        )

# 中文注释：任何失败请求都阻止最终导出，避免把不完整排序误当正式结果。
missing_ids = [int(sample["sample_id"]) for sample in samples if int(sample["sample_id"]) not in completed]
if missing_ids:
    raise RuntimeError(f"仍有 {len(missing_ids)} 个样本失败，请检查 checkpoint 末尾 error 后重跑；首个 sample_id={missing_ids[0]}")

# 中文注释：整理排名、计算 Accuracy@1 并写入独立结果工作簿。
final_records = [completed[int(sample["sample_id"])] for sample in samples]
output_frame = build_output_frame(final_records, terms)
accuracy = {"accuracy_at_1": float(output_frame["hit_at_1"].mean())}
graph_rerank_sample_count = int(output_frame["graph_rerank_applied"].sum())
graph_changed_top1_count = int(
    (
        output_frame["graph_rerank_applied"]
        & (output_frame["initial_top1_id"] != output_frame["predicted_top1_id"])
    ).sum()
)
no_match_count = int((output_frame["prediction_status"] == "no_match").sum())
graph_output_suffix = "_hpo-graph" if hpo_graph_enabled else ""
output_path = RERANK_RESULTS_DIR / f"{DATASET_KEY}_llm_rerank_candidates20_{embedding_tag}_{llm_tag}{graph_output_suffix}.xlsx"
run_config = {
    "dataset_key": DATASET_KEY,
    "sample_count": len(output_frame),
    "ontology": ontology_key,
    "accepted_gold_policy": "candidate hits any ID in accepted_gold_ids_json",
    "precompute_path": str(precompute_path),
    "precompute_sha256": precompute_hash,
    "embedding_model": resolved_embedding_model,
    "oar_enabled": use_oar,
    "llm_model": resolved_llm_model,
    "llm_temperature": LLM_TEMPERATURE,
    "llm_candidate_count": LLM_CANDIDATE_COUNT,
    "hpo_graph_rerank_enabled": hpo_graph_enabled,
    "hpo_graph_path": str(HPO_GRAPH_PATH) if hpo_graph_enabled else "",
    "hpo_graph_sha256": hpo_graph_sha256,
    "hpo_graph_probe_top_k": HPO_GRAPH_PROBE_TOP_K,
    "hpo_max_ancestor_distance": HPO_MAX_ANCESTOR_DISTANCE,
    "hpo_graph_trigger_rule": HPO_GRAPH_TRIGGER_RULE,
    "graph_rerank_sample_count": graph_rerank_sample_count,
    "graph_changed_top1_count": graph_changed_top1_count,
    "no_match_count": no_match_count,
    "rerank_config_hash": rerank_config_hash,
    "rerank_config_identity": rerank_config_identity,
    "llm_max_concurrency": resolved_concurrency,
    "llm_requests_per_minute": LLM_REQUESTS_PER_MINUTE,
    "max_attempts": MAX_ATTEMPTS,
    "checkpoint_path": str(checkpoint_path),
    "embedding_api_calls_this_notebook": 0,
    "elapsed_seconds_this_run": round(time.perf_counter() - started_at, 3),
    "graph_reranks_this_run": graph_reranks_this_run,
    **accuracy,
}
export_reranking(output_path, output_frame, run_config)
print(f"saved={output_path}")
print(accuracy)